# Let's build the GPT Tokenizer（完整课堂版）

对照 Karpathy [minbpe](https://github.com/karpathy/minbpe) / Lecture 8。  
本 notebook **自包含**：helpers + `BasicTokenizer`（byte-level BPE）+ `RegexTokenizer`（GPT-2/4 切分 + 特殊 token）。

- 视频：https://www.youtube.com/watch?v=zduSFxRajkE
- 官方 Colab：https://colab.research.google.com/drive/1y0KnCFZvGVf_odSfcNAws6kcDD7HsI0L
- 笔记：Vault `nanoGPT · Let's build the GPT Tokenizer`

阅读顺序建议：先跑通 Section A/B，再看 C（regex 版）。看不懂的地方直接问。

## A. 公共工具：`get_stats` / `merge`

In [ ]:
def get_stats(ids, counts=None):
    """统计相邻 pair 出现次数。ids: list[int]"""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """把 ids 里所有相邻 pair 替换成新 id。"""
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids


# 快速自测
assert get_stats([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
assert merge([1, 2, 3, 1, 2], (1, 2), 4) == [4, 3, 4]
print('helpers ok')

## B. `BasicTokenizer`：byte-level BPE

要点：

1. 文本先 `encode('utf-8')` 变成 bytes（0..255）
2. 词表从 256 个 byte 起步
3. 训练：合并 `vocab_size - 256` 次
4. encode：按 merges **学到的先后顺序**（merge index 更小的优先）反复合并
5. decode：id → bytes 拼接 → UTF-8 解码

In [ ]:
class BasicTokenizer:
    """最小 byte-level BPE（对齐 minbpe BasicTokenizer）。"""

    def __init__(self):
        self.merges = {}  # (int, int) -> int
        self.vocab = {idx: bytes([idx]) for idx in range(256)}  # int -> bytes

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)  # 0..255

        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = merge(ids, pair, idx)
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r}) count={stats[pair]}")

        self.merges = merges
        self.vocab = vocab

    def decode(self, ids):
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            # 优先应用“更早学到”的 merge（id 更小）
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids


print('BasicTokenizer defined')

In [ ]:
# 用一段稍长的文本训练并往返验证
train_text = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?
""".strip()

tok = BasicTokenizer()
tok.train(train_text, vocab_size=300, verbose=True)  # 256 + 44 merges

s = "hear me speak"
ids = tok.encode(s)
print('text:', repr(s))
print('ids:', ids)
print('decoded:', repr(tok.decode(ids)))
print('roundtrip:', tok.decode(tok.encode(train_text)) == train_text)
print('compression on train_text:', len(train_text.encode()), 'bytes ->', len(tok.encode(train_text)), 'tokens')
print('num merges:', len(tok.merges), 'vocab size:', len(tok.vocab))

## C. `RegexTokenizer`：先正则切分，再在块内做 BPE

GPT-2/4 不会把整篇文档当成一条超长 byte 串硬 merge，而是：

1. 用正则把文本切成许多 chunk（词、数字、空白等）
2. **每个 chunk 内部**各自做 BPE（跨 chunk 不合并）
3. 可选：注册特殊 token（如 `<|endoftext|>`）

依赖：`regex`（支持 `\p{L}` 等 Unicode 属性）。本仓库用 uv：

```bash
uv add regex
```

In [ ]:
# 本仓库已用 `uv add regex` 装好；若换环境再执行: uv add regex
import regex as re

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

print('sample GPT4 chunks:')
print(re.findall(GPT4_SPLIT_PATTERN, "Hello, world! It's 2024.\nNext"))

In [ ]:
class RegexTokenizer:
    """对齐 minbpe RegexTokenizer：regex 预切分 + byte-level BPE + special tokens。"""

    def __init__(self, pattern=None):
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        self.special_tokens = {}
        self.inverse_special_tokens = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        text_chunks = re.findall(self.compiled_pattern, text)
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]

        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            stats = {}
            for chunk_ids in ids:
                get_stats(chunk_ids, stats)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r}) count={stats[pair]}")

        self.merges = merges
        self.vocab = vocab

    def register_special_tokens(self, special_tokens):
        self.special_tokens = dict(special_tokens)
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}

    def decode(self, ids):
        parts = []
        for idx in ids:
            if idx in self.vocab:
                parts.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                parts.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"invalid token id: {idx}")
        return b"".join(parts).decode("utf-8", errors="replace")

    def _encode_chunk(self, text_bytes):
        ids = list(text_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def encode_ordinary(self, text):
        text_chunks = re.findall(self.compiled_pattern, text)
        ids = []
        for chunk in text_chunks:
            ids.extend(self._encode_chunk(chunk.encode("utf-8")))
        return ids

    def encode(self, text, allowed_special="none_raise"):
        if allowed_special == "all":
            special = self.special_tokens
        elif allowed_special == "none":
            special = {}
        elif allowed_special == "none_raise":
            special = {}
            assert all(token not in text for token in self.special_tokens)
        elif isinstance(allowed_special, set):
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"allowed_special={allowed_special} not understood")

        if not special:
            return self.encode_ordinary(text)

        special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"
        special_chunks = re.split(special_pattern, text)
        ids = []
        for part in special_chunks:
            if part in special:
                ids.append(special[part])
            else:
                ids.extend(self.encode_ordinary(part))
        return ids


print('RegexTokenizer defined')

In [ ]:
rtok = RegexTokenizer()
rtok.train(train_text, vocab_size=300, verbose=False)
rtok.register_special_tokens({"<|endoftext|>": 300})

s = "Speak, speak.<|endoftext|>All:"
ids = rtok.encode(s, allowed_special="all")
print('text:', repr(s))
print('ids:', ids)
print('decoded:', repr(rtok.decode(ids)))

# 对比：同样词表预算下，Basic vs Regex 的切分差异
b_ids = tok.encode("Hello world!")
r_ids = rtok.encode("Hello world!", allowed_special="none")
print('Basic  Hello world! ->', b_ids)
print('Regex  Hello world! ->', r_ids)
print('Regex chunks:', re.findall(GPT4_SPLIT_PATTERN, "Hello world!"))

## D. 阅读时重点对照

| 主题 | 看哪里 |
|------|--------|
| 为什么从 256 bytes 起 | `BasicTokenizer.train` 开头 |
| 重写序列 | `ids = merge(...)` |
| encode 为何按 merge 顺序 | `min(..., merges.get(p, inf))` |
| decode | `vocab[id]` 拼 bytes |
| regex 预切分 | `re.findall(pattern, text)` |
| 特殊 token | `register_special_tokens` + `encode(..., allowed_special="all")` |

常见坑：

- encode 不是简单查表，而是在当前序列上**反复**找可合并 pair
- regex 版：**跨 chunk 不会 merge**（空格+词 常被切开）
- 不完整 UTF-8 字节序列 decode 时用 `errors='replace'`

看完后把疑问标出行号/函数名问我即可。